<a href="https://colab.research.google.com/github/guilhermefrrr/ucl2425/blob/main/python-notebooks/field-tilt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Presentation

This Python code is part of a football data analysis project, focusing on extracting and processing individual player statistics. It utilizes the pandas library to retrieve data from the FBref website, parsing HTML tables from match pages and organizing them into a structured format.

The project comprises three main steps using Python:

1. **Extracting stats:** This step is the core of the project. It extracts a wide range of player statistics, encompassing passing, shooting, and defensive actions. This data is then consolidated into a single comprehensive DataFrame that captures all individual player performances throughout the competition.
2. **Field Tilt (this code):** Utilizing the DataFrame created in the previous step, this part of the project calculates the Field Tilt for each club across the competition. Field Tilt is a metric that quantifies a team's territorial dominance during a match, often estimated by comparing possession percentages or the location of attacking actions. A higher Field Tilt indicates greater control over the game.
3. **Extracting fixtures:** This step focuses on gathering general information about each match in the competition. It extracts details such as the date and location of each game, providing context for the individual player statistics and Field Tilt calculations.

All the data collected across these three steps is then integrated into a Power BI report. This report leverages the extracted statistics and calculated metrics to provide interactive dashboards and visualizations, enabling in-depth analysis of player and team performance throughout the competition.

The code in this file (written below) iterates through a list of match URLs, extracting the relevant information and introducing a 10-second delay between requests to the FBref server to avoid overloading and potential blocking. Maintaining this delay function is crucial for the smooth operation of the process.

*The project is intended for educational and personal use only. FBref has its own terms of service and usage policies that should be respected. Please use this project responsibly and ethically.*

## Libraries

In [ ]:
import pandas as pd
from google.colab import drive

## Variables

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/guilhermefrrr/ucl2425/refs/heads/main/complete-data/statsucl2425.csv')

In [ ]:
# Information about the file that will be created with the data extracted
file_name = 'all_data.csv'
folder_path = '/content/drive/MyDrive/'

file_path = folder_path + file_name

## Function

In [ ]:
def calculate_field_tilt(df):
    """
    Calculates the Field Tilt for each team in each match and the accumulated Field Tilt up to the current matchday.
    """

    # Calculates the Field Tilt by match

    # Groups the data by match and team, calculating the total touches in the final third
    touches_by_match_team = df.groupby(['Match_ID', 'Squad', 'MD'])['Touches_Att 3rd'].sum().reset_index()
    # Calculates the total number of touches for each match (adding up the touches for both teams)
    total_touches_by_match = touches_by_match_team.groupby(['Match_ID', 'MD'])['Touches_Att 3rd'].sum().reset_index()
    # Renames the column to avoid merge conflicts
    total_touches_by_match = total_touches_by_match.rename(columns={'Touches_Att 3rd': 'Total_Touches_Att 3rd'})
    # Merges the data to have the total touches and the team's touches in the same row
    field_tilt_df = pd.merge(touches_by_match_team, total_touches_by_match, on=['Match_ID', 'MD'])
    # Calculates the Field Tilt
    field_tilt_df['Field_Tilt'] = field_tilt_df['Touches_Att 3rd'] / field_tilt_df['Total_Touches_Att 3rd']

    # Calculates the cumulative Field Tilt up to the current matchday

    # Calculates the cumulative sum of touches in the attacking third for each squad
    field_tilt_df['Cum_Touches_Att 3rd'] = field_tilt_df.groupby('Squad')['Touches_Att 3rd'].expanding().sum().reset_index(level=0, drop=True)
    # Calculates the cumulative sum of touches in the attacking third for each match that the squad was part of
    field_tilt_df['Cum_Total_Touches_Att 3rd'] = field_tilt_df.groupby('Squad')['Total_Touches_Att 3rd'].expanding().sum().reset_index(level=0, drop=True)
    # Calculates the cumulative Field Tilt
    field_tilt_df['Field_Tilt_Over_Matches'] = field_tilt_df['Cum_Touches_Att 3rd'] / field_tilt_df['Cum_Total_Touches_Att 3rd']


    return field_tilt_df

## Processing data

In [ ]:
ucl2425_fieldtilt = calculate_field_tilt(df)
ucl2425_fieldtilt.head()

,Match_ID,Squad,MD,Touches_Att 3rd,Total_Touches_Att 3rd,Field_Tilt,Cum_Touches_Att 3rd,Cum_Total_Touches_Att 3rd,Field_Tilt_Over_Matches
0,1,Aston Villa,1,162,258,0.627907,162.0,258.0,0.627907
1,1,Young Boys,1,96,258,0.372093,96.0,258.0,0.372093
2,2,Juventus,1,89,309,0.288026,89.0,309.0,0.288026
3,2,PSV Eindhoven,1,220,309,0.711974,220.0,309.0,0.711974
4,3,Lille,1,119,233,0.510730,119.0,233.0,0.510730


In [ ]:
calculate_field_tilt(df)[['Match_ID', 'Squad', 'Field_Tilt']].sort_values(by=['Field_Tilt'], ascending=False).head()

,Match_ID,Squad,Field_Tilt
48,25,Manchester City,0.946640
96,49,Manchester City,0.918200
258,130,Bayern Munich,0.910180
250,126,Bayern Munich,0.902985
10,6,Bayern Munich,0.889213


In [ ]:
# Multiplies the columns by 100 to get the values in percentage
ucl2425_fieldtilt['Field_Tilt'] = ucl2425_fieldtilt['Field_Tilt'] * 100
ucl2425_fieldtilt['Field_Tilt_Over_Matches'] = ucl2425_fieldtilt['Field_Tilt_Over_Matches'] * 100

# Renames the columns including "%" in the name
ucl2425_fieldtilt = ucl2425_fieldtilt.rename(columns={
    'Field_Tilt': 'Field_Tilt%',
    'Field_Tilt_Over_Matches': 'Field_Tilt_Over_Matches%'
})

# Rounds the values to two decimal places, keeping the numeric type
ucl2425_fieldtilt['Field_Tilt%'] = ucl2425_fieldtilt['Field_Tilt%'].round(2)
ucl2425_fieldtilt['Field_Tilt_Over_Matches%'] = ucl2425_fieldtilt['Field_Tilt_Over_Matches%'].round(2)

In [ ]:
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns

In [ ]:
ucl2425_fieldtilt.sort_values(by=['Field_Tilt%'], ascending=False).head(10)

,Match_ID,Squad,MD,Touches_Att 3rd,Total_Touches_Att 3rd,Field_Tilt%,Cum_Touches_Att 3rd,Cum_Total_Touches_Att 3rd,Field_Tilt_Over_Matches%
48,25,Manchester City,2,479,506,94.66,800.0,881.0,90.81
96,49,Manchester City,3,449,489,91.82,1249.0,1370.0,91.17
258,130,Bayern Munich,8,456,501,91.02,2381.0,2843.0,83.75
250,126,Bayern Munich,7,363,402,90.30,1925.0,2342.0,82.19
10,6,Bayern Munich,1,305,343,88.92,305.0,343.0,88.92
112,57,Manchester City,4,281,323,87.00,1530.0,1693.0,90.37
253,127,Manchester City,8,370,426,86.85,2541.0,3102.0,81.91
132,67,Bayern Munich,4,377,435,86.67,1111.0,1320.0,84.17
92,47,Atalanta,3,390,451,86.47,777.0,1040.0,74.71
42,22,Barcelona,2,269,312,86.22,327.0,556.0,58.81


## Saving

In [ ]:
# Mount Google Drive to save file (if necessary)
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#ucl2425_fieldtilt.to_csv(file_path, index=False)